# RAG V12 — Multi-Document RAG — Stage 2\n\nRun the single code cell below. It contains the complete Stage 2 implementation and validation: 27 PDFs → 11,502 chunks → MiniLM embeddings → Pinecone V12 index → metadata-filtered retrieval.

In [ ]:
from collections import Counter
from pathlib import Path
import os
import re
import time

from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pinecone import DenseVectorQuery, Pinecone, SchemaBuilder
from sentence_transformers import SentenceTransformer


# ============================================================
# RAG V12 - MULTI-DOCUMENT RAG
# JUPYTER NOTEBOOK - STAGE 2
#
# COMPLETE TEST:
# 1. Find the repo and 27 PDFs
# 2. Load + clean + chunk all PDFs
# 3. Create provider/document/page metadata
# 4. Create MiniLM embeddings
# 5. Create a NEW Pinecone V12 index
# 6. Upload all chunks
# 7. Validate semantic search
# 8. Validate provider filters
# 9. Validate document filters
# 10. Validate document-safe neighbors
#
# IMPORTANT:
# This does NOT modify the V11 Pinecone index.
# ============================================================


# ============================================================
# CONFIGURATION
# ============================================================

REPO_NAME = "kantinstephane10-rag-from-scratch-azure-aws"

CURRENT_DIRECTORY = Path.cwd()

repo_candidates = [
    CURRENT_DIRECTORY,
    CURRENT_DIRECTORY / REPO_NAME,
    CURRENT_DIRECTORY.parent / REPO_NAME,
    CURRENT_DIRECTORY.parent.parent / REPO_NAME,
]

REPO_ROOT = None

for candidate in repo_candidates:
    cloud_folder = candidate / "data" / "cloud_documents"

    if cloud_folder.exists():
        REPO_ROOT = candidate
        break

if REPO_ROOT is None:
    raise FileNotFoundError(
        "Could not find the repository and data/cloud_documents."
    )

CLOUD_DOCUMENT_ROOT = (
    REPO_ROOT
    / "data"
    / "cloud_documents"
)

EXPECTED_PROVIDERS = [
    "AWS",
    "Azure",
    "GCP",
    "Oracle",
    "IBM",
]

EXPECTED_DOCUMENT_COUNTS = {
    "AWS": 8,
    "Azure": 5,
    "GCP": 5,
    "Oracle": 4,
    "IBM": 5,
}

EXPECTED_TOTAL_PDFS = 27

CHUNK_SIZE = 800
CHUNK_OVERLAP = 120
NEIGHBOR_RANGE = 1

EMBEDDING_MODEL = (
    "sentence-transformers/"
    "all-MiniLM-L6-v2"
)

PINECONE_INDEX_NAME = os.getenv(
    "PINECONE_V12_INDEX_NAME",
    "multicloud-rag-v12",
)

PINECONE_NAMESPACE = os.getenv(
    "PINECONE_V12_NAMESPACE",
    "cloud-documents",
)

PINECONE_VECTOR_FIELD = "embedding"
PINECONE_DIMENSION = 384

PINECONE_API_KEY = os.getenv(
    "PINECONE_API_KEY"
)


print("=" * 70)
print("V12 STAGE 2 CONFIGURATION")
print("=" * 70)

print("Repository:")
print(REPO_ROOT.resolve())

print()
print("Cloud corpus:")
print(CLOUD_DOCUMENT_ROOT.resolve())

print()
print("Pinecone index:", PINECONE_INDEX_NAME)
print("Pinecone namespace:", PINECONE_NAMESPACE)
print("Embedding model:", EMBEDDING_MODEL)


# ============================================================
# API KEY CHECK
# ============================================================

if not PINECONE_API_KEY:
    raise RuntimeError(
        "PINECONE_API_KEY is not available in this Jupyter kernel. "
        "Set the environment variable before starting Jupyter/VS Code."
    )

print()
print("Pinecone API key available: PASS ✅")


# ============================================================
# TEXT CLEANING
# ============================================================

def clean_cloud_text(text):
    text = text.replace(
        "\t",
        " ",
    )

    text = re.sub(
        r"(?m)^\s*\d{1,4}\s*$",
        "",
        text,
    )

    text = re.sub(
        r"[ ]+",
        " ",
        text,
    )

    text = re.sub(
        r"\n{3,}",
        "\n\n",
        text,
    )

    return text.strip()


# ============================================================
# METADATA HELPERS
# ============================================================

def slugify(text):
    value = text.lower()

    value = re.sub(
        r"[^a-z0-9]+",
        "-",
        value,
    )

    return value.strip("-")


def title_from_filename(pdf_path):
    title = pdf_path.stem

    title = re.sub(
        r"[_-]+",
        " ",
        title,
    )

    title = re.sub(
        r"\s+",
        " ",
        title,
    )

    return title.strip()


def get_provider_from_path(pdf_path):
    relative_path = pdf_path.relative_to(
        CLOUD_DOCUMENT_ROOT
    )

    provider = relative_path.parts[0]

    if provider not in EXPECTED_PROVIDERS:
        raise ValueError(
            f"Unknown provider folder: {provider}"
        )

    return provider


# ============================================================
# DISCOVER THE 27 PDFs
# ============================================================

def discover_cloud_pdfs():
    pdf_files = sorted(
        path
        for path in CLOUD_DOCUMENT_ROOT.rglob(
            "*.pdf"
        )
        if path.is_file()
    )

    records = []

    for pdf_path in pdf_files:
        provider = get_provider_from_path(
            pdf_path
        )

        document_id = (
            f"{provider.lower()}__"
            f"{slugify(pdf_path.stem)}"
        )

        records.append(
            {
                "path":
                    pdf_path,

                "provider":
                    provider,

                "document_id":
                    document_id,

                "document_title":
                    title_from_filename(
                        pdf_path
                    ),

                "source":
                    pdf_path.name,
            }
        )

    return records


pdf_records = discover_cloud_pdfs()

provider_document_counts = Counter(
    record["provider"]
    for record in pdf_records
)


print()
print("=" * 70)
print("DOCUMENT DISCOVERY")
print("=" * 70)

print(
    "PDFs discovered:",
    len(pdf_records),
)

for provider in EXPECTED_PROVIDERS:
    print(
        f"{provider:<8}",
        provider_document_counts[
            provider
        ],
    )


# ============================================================
# LOAD ALL PDF PAGES
# ============================================================

def load_cloud_documents(
    pdf_records,
):
    all_pages = []
    provider_page_counts = Counter()

    for document_number, record in enumerate(
        pdf_records,
        start=1,
    ):
        print(
            f"[{document_number}/"
            f"{len(pdf_records)}] "
            f"{record['provider']} - "
            f"{record['source']}"
        )

        loader = PyMuPDFLoader(
            str(
                record["path"]
            )
        )

        pages = loader.load()

        for page in pages:
            page.page_content = (
                clean_cloud_text(
                    page.page_content
                )
            )

            pdf_page = (
                int(
                    page.metadata.get(
                        "page",
                        0,
                    )
                )
                + 1
            )

            page.metadata[
                "pdf_page"
            ] = pdf_page

            page.metadata[
                "provider"
            ] = record[
                "provider"
            ]

            page.metadata[
                "document_id"
            ] = record[
                "document_id"
            ]

            page.metadata[
                "document_title"
            ] = record[
                "document_title"
            ]

            page.metadata[
                "source"
            ] = record[
                "source"
            ]

            page.metadata[
                "source_path"
            ] = str(
                record[
                    "path"
                ]
            )

            provider_page_counts[
                record[
                    "provider"
                ]
            ] += 1

        all_pages.extend(
            pages
        )

    return (
        all_pages,
        provider_page_counts,
    )


print()
print("=" * 70)
print("LOADING CLOUD DOCUMENTS")
print("=" * 70)

pages, provider_page_counts = (
    load_cloud_documents(
        pdf_records
    )
)


print()
print(
    "Total pages loaded:",
    len(pages),
)


# ============================================================
# CHUNK ALL DOCUMENTS
# ============================================================

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=[
        "\n\n",
        "\n",
        ". ",
        " ",
        "",
    ],
)

chunks = splitter.split_documents(
    pages
)

chunks = [
    chunk
    for chunk in chunks
    if chunk.page_content.strip()
]


per_document_chunk_counter = Counter()

for global_chunk_number, chunk in enumerate(
    chunks
):
    document_id = chunk.metadata[
        "document_id"
    ]

    document_chunk_number = (
        per_document_chunk_counter[
            document_id
        ]
    )

    chunk.metadata[
        "document_chunk_number"
    ] = document_chunk_number

    chunk.metadata[
        "global_chunk_number"
    ] = global_chunk_number

    chunk.metadata[
        "chunk_characters"
    ] = len(
        chunk.page_content
    )

    chunk.metadata[
        "chunk_id"
    ] = (
        f"{document_id}"
        "__chunk_"
        f"{document_chunk_number:05d}"
    )

    per_document_chunk_counter[
        document_id
    ] += 1


print()
print("=" * 70)
print("CHUNKING RESULT")
print("=" * 70)

print(
    "Pages:",
    len(pages),
)

print(
    "Chunks:",
    len(chunks),
)

print(
    "Documents represented:",
    len(
        per_document_chunk_counter
    ),
)


# ============================================================
# BUILD DOCUMENT-SAFE LOOKUP
# ============================================================

rag_chunk_lookup = {}

for chunk in chunks:
    key = (
        chunk.metadata[
            "document_id"
        ],
        chunk.metadata[
            "document_chunk_number"
        ],
    )

    rag_chunk_lookup[
        key
    ] = chunk


# ============================================================
# DOCUMENT-SAFE NEIGHBORS
# ============================================================

def get_document_neighbors(
    document_id,
    document_chunk_number,
    neighbor_range=1,
):
    neighbors = []

    for offset in range(
        -neighbor_range,
        neighbor_range + 1,
    ):
        key = (
            document_id,
            document_chunk_number
            + offset,
        )

        neighbor = (
            rag_chunk_lookup.get(
                key
            )
        )

        if neighbor is not None:
            neighbors.append(
                neighbor
            )

    return neighbors


# ============================================================
# LOAD EMBEDDING MODEL
# ============================================================

print()
print("=" * 70)
print("LOADING EMBEDDING MODEL")
print("=" * 70)

embedding_model = SentenceTransformer(
    EMBEDDING_MODEL
)

print(
    "Embedding model loaded ✅"
)


# ============================================================
# CREATE EMBEDDINGS
# ============================================================

chunk_texts = [
    chunk.page_content
    for chunk in chunks
]


print()
print("=" * 70)
print("CREATING EMBEDDINGS")
print("=" * 70)

embeddings = embedding_model.encode(
    chunk_texts,
    show_progress_bar=True,
    normalize_embeddings=True,
)


print()
print(
    "Embedding shape:",
    embeddings.shape,
)


if (
    embeddings.shape[1]
    != PINECONE_DIMENSION
):
    raise RuntimeError(
        "Embedding dimension mismatch. "
        f"Expected {PINECONE_DIMENSION}, "
        f"got {embeddings.shape[1]}."
    )


# ============================================================
# CONNECT TO PINECONE
# ============================================================

pinecone_client = Pinecone(
    api_key=PINECONE_API_KEY
)


# ============================================================
# CREATE OR VALIDATE V12 INDEX
# ============================================================

print()
print("=" * 70)
print("PINECONE INDEX")
print("=" * 70)


if not pinecone_client.indexes.exists(
    PINECONE_INDEX_NAME
):
    print(
        "Creating Pinecone index:",
        PINECONE_INDEX_NAME,
    )

    schema = (
        SchemaBuilder()
        .add_dense_vector_field(
            PINECONE_VECTOR_FIELD,
            dimension=
                PINECONE_DIMENSION,
            metric="cosine",
        )
        .build()
    )

    pinecone_client.indexes.create(
        name=
            PINECONE_INDEX_NAME,
        schema=
            schema,
    )

    print(
        "Pinecone V12 index created ✅"
    )

else:
    print(
        "Pinecone V12 index already exists:",
        PINECONE_INDEX_NAME,
    )


index_info = (
    pinecone_client
    .indexes
    .describe(
        PINECONE_INDEX_NAME
    )
)


vector_field = (
    index_info
    .schema
    .fields[
        PINECONE_VECTOR_FIELD
    ]
)


if (
    int(
        vector_field.dimension
    )
    !=
    PINECONE_DIMENSION
):
    raise RuntimeError(
        "Existing Pinecone V12 index "
        "has the wrong dimension."
    )


index = pinecone_client.index(
    PINECONE_INDEX_NAME
)


# ============================================================
# BUILD PINECONE DOCUMENTS
# ============================================================

pinecone_documents = []


for i, chunk in enumerate(
    chunks
):
    metadata = chunk.metadata

    pinecone_documents.append(
        {
            "_id":
                metadata[
                    "chunk_id"
                ],

            PINECONE_VECTOR_FIELD:
                embeddings[
                    i
                ].tolist(),

            "text":
                chunk.page_content,

            "provider":
                metadata[
                    "provider"
                ],

            "document_id":
                metadata[
                    "document_id"
                ],

            "document_title":
                metadata[
                    "document_title"
                ],

            "source":
                metadata[
                    "source"
                ],

            "pdf_page":
                metadata[
                    "pdf_page"
                ],

            "document_chunk_number":
                metadata[
                    "document_chunk_number"
                ],

            "global_chunk_number":
                metadata[
                    "global_chunk_number"
                ],
        }
    )


print()
print(
    "Pinecone documents prepared:",
    len(
        pinecone_documents
    ),
)


# ============================================================
# UPSERT ALL 11,502 CHUNKS
# ============================================================

print()
print("=" * 70)
print("UPLOADING TO PINECONE")
print("=" * 70)


upsert_result = (
    index.documents.batch_upsert(
        namespace=
            PINECONE_NAMESPACE,
        documents=
            pinecone_documents,
    )
)


if upsert_result.errors:
    print(
        "Pinecone upsert errors:"
    )

    for error in (
        upsert_result.errors
    ):
        print(
            error.batch_index,
            error.error_message,
        )

    raise RuntimeError(
        "Pinecone batch upsert failed."
    )


print(
    "Pinecone batch upsert completed ✅"
)


# ============================================================
# WAIT FOR PINECONE CONSISTENCY
# ============================================================

def wait_for_documents(
    document_ids,
    timeout_seconds=120,
):
    deadline = (
        time.time()
        +
        timeout_seconds
    )

    while time.time() < deadline:
        response = (
            index.documents.fetch(
                namespace=
                    PINECONE_NAMESPACE,
                ids=
                    document_ids,
            )
        )

        if (
            len(
                response.documents
            )
            ==
            len(
                document_ids
            )
        ):
            return True

        time.sleep(
            2
        )

    return False


sample_ids = [
    chunks[0].metadata[
        "chunk_id"
    ],
    chunks[
        len(chunks) // 2
    ].metadata[
        "chunk_id"
    ],
    chunks[-1].metadata[
        "chunk_id"
    ],
]


documents_ready = (
    wait_for_documents(
        sample_ids
    )
)


if not documents_ready:
    raise TimeoutError(
        "Pinecone documents did not become "
        "available within the timeout."
    )


print(
    "Pinecone documents are available ✅"
)


# ============================================================
# SEARCH HELPER
# ============================================================

def search_pinecone(
    question,
    top_k=5,
    provider=None,
    document_id=None,
):
    query_embedding = (
        embedding_model.encode(
            [question],
            normalize_embeddings=True,
        )[0]
    )

    filters = []

    if provider is not None:
        filters.append(
            {
                "provider": {
                    "$eq":
                        provider
                }
            }
        )

    if document_id is not None:
        filters.append(
            {
                "document_id": {
                    "$eq":
                        document_id
                }
            }
        )

    if len(filters) == 0:
        metadata_filter = None

    elif len(filters) == 1:
        metadata_filter = (
            filters[0]
        )

    else:
        metadata_filter = {
            "$and":
                filters
        }

    results = (
        index.documents.search(
            namespace=
                PINECONE_NAMESPACE,

            top_k=
                top_k,

            score_by=[
                DenseVectorQuery(
                    field=
                        PINECONE_VECTOR_FIELD,
                    values=
                        query_embedding.tolist(),
                )
            ],

            include_fields=[
                "text",
                "provider",
                "document_id",
                "document_title",
                "source",
                "pdf_page",
                "document_chunk_number",
                "global_chunk_number",
            ],

            filter=
                metadata_filter,
        )
    )

    matches = []

    for match in results.matches:
        matches.append(
            {
                "score":
                    float(
                        match.score
                    ),

                "text":
                    match.get(
                        "text",
                        "",
                    ),

                "provider":
                    match.get(
                        "provider",
                        "",
                    ),

                "document_id":
                    match.get(
                        "document_id",
                        "",
                    ),

                "document_title":
                    match.get(
                        "document_title",
                        "",
                    ),

                "source":
                    match.get(
                        "source",
                        "",
                    ),

                "pdf_page":
                    int(
                        match.get(
                            "pdf_page",
                            0,
                        )
                    ),

                "document_chunk_number":
                    int(
                        match.get(
                            "document_chunk_number",
                            0,
                        )
                    ),

                "global_chunk_number":
                    int(
                        match.get(
                            "global_chunk_number",
                            0,
                        )
                    ),
            }
        )

    return matches


# ============================================================
# SHOW ONE GENERAL SEMANTIC SEARCH
# ============================================================

print()
print("=" * 70)
print("GENERAL MULTI-DOCUMENT SEARCH")
print("=" * 70)


general_results = (
    search_pinecone(
        "How should cloud workloads be designed for security and reliability?",
        top_k=5,
    )
)


for i, result in enumerate(
    general_results,
    start=1,
):
    print()
    print(
        f"Result {i}"
    )

    print(
        "Score:",
        round(
            result[
                "score"
            ],
            4,
        ),
    )

    print(
        "Provider:",
        result[
            "provider"
        ],
    )

    print(
        "Document:",
        result[
            "document_title"
        ],
    )

    print(
        "PDF page:",
        result[
            "pdf_page"
        ],
    )

    print(
        result[
            "text"
        ][
            :500
        ]
    )


# ============================================================
# PROVIDER FILTER SEARCHES
# ============================================================

provider_test_queries = {
    "AWS":
        "security architecture identity access management",

    "Azure":
        "reliability architecture cloud workloads",

    "GCP":
        "cloud security infrastructure trust",

    "Oracle":
        "identity access management network security",

    "IBM":
        "hybrid cloud security architecture",
}


provider_filter_results = {}


print()
print("=" * 70)
print("PROVIDER FILTER TEST SEARCHES")
print("=" * 70)


for provider, question in (
    provider_test_queries.items()
):
    results = search_pinecone(
        question,
        top_k=3,
        provider=provider,
    )

    provider_filter_results[
        provider
    ] = results

    print()
    print(
        provider,
        "results:",
        len(results),
    )

    for result in results:
        print(
            " -",
            result[
                "provider"
            ],
            "|",
            result[
                "source"
            ],
            "| page",
            result[
                "pdf_page"
            ],
        )


# ============================================================
# DOCUMENT FILTER TEST
# ============================================================

AWS_SECURITY_DOCUMENT_ID = (
    "aws__aws-security-reference-architecture"
)


document_filter_results = (
    search_pinecone(
        "security architecture identity logging network protection",
        top_k=3,
        document_id=
            AWS_SECURITY_DOCUMENT_ID,
    )
)


print()
print("=" * 70)
print("DOCUMENT FILTER TEST")
print("=" * 70)


for result in document_filter_results:
    print(
        result[
            "document_id"
        ],
        "| page",
        result[
            "pdf_page"
        ],
    )


# ============================================================
# DOCUMENT-SAFE NEIGHBOR TEST FROM PINECONE RESULT
# ============================================================

neighbor_test_result = (
    document_filter_results[0]
)


retrieved_neighbors = (
    get_document_neighbors(
        document_id=
            neighbor_test_result[
                "document_id"
            ],

        document_chunk_number=
            neighbor_test_result[
                "document_chunk_number"
            ],

        neighbor_range=
            NEIGHBOR_RANGE,
    )
)


print()
print("=" * 70)
print("DOCUMENT-SAFE NEIGHBORS")
print("=" * 70)


for neighbor in retrieved_neighbors:
    print(
        neighbor.metadata[
            "provider"
        ],
        "|",
        neighbor.metadata[
            "document_id"
        ],
        "| chunk",
        neighbor.metadata[
            "document_chunk_number"
        ],
        "| page",
        neighbor.metadata[
            "pdf_page"
        ],
    )


# ============================================================
# VALIDATION
# ============================================================

test_results = []


def check(
    condition,
    label,
):
    if condition:
        print(
            f"{label}: PASS ✅"
        )

        test_results.append(
            True
        )

    else:
        print(
            f"{label}: FAIL ❌"
        )

        test_results.append(
            False
        )


print()
print("=" * 70)
print("RAG V12 STAGE 2 VALIDATION")
print("=" * 70)


# TEST 1
check(
    PINECONE_API_KEY
    is not None,
    "Pinecone API key available",
)


# TEST 2
check(
    len(pdf_records)
    ==
    EXPECTED_TOTAL_PDFS,
    "27 PDFs loaded",
)


# TEST 3
check(
    len(chunks)
    ==
    11502,
    "Expected 11,502 chunks reproduced",
)


# TEST 4
check(
    embeddings.shape[0]
    ==
    len(chunks),
    "One embedding created per chunk",
)


# TEST 5
check(
    embeddings.shape[1]
    ==
    PINECONE_DIMENSION,
    "Embedding dimension = 384",
)


# TEST 6
check(
    pinecone_client.indexes.exists(
        PINECONE_INDEX_NAME
    ),
    "Pinecone V12 index exists",
)


# TEST 7
check(
    PINECONE_VECTOR_FIELD
    in
    index_info.schema.fields,
    "Dense vector field exists",
)


# TEST 8
check(
    int(
        vector_field.dimension
    )
    ==
    PINECONE_DIMENSION,
    "Pinecone dimension = 384",
)


# TEST 9
check(
    str(
        vector_field.metric
    ).lower()
    ==
    "cosine",
    "Pinecone metric = cosine",
)


# TEST 10
check(
    documents_ready,
    "Pinecone documents persisted",
)


# TEST 11
check(
    len(
        general_results
    )
    > 0,
    "Multi-document semantic search returns results",
)


# TEST 12
check(
    all(
        result[
            "provider"
        ]
        in
        EXPECTED_PROVIDERS

        for result
        in general_results
    ),
    "Semantic search restores provider metadata",
)


# TESTS 13-17
for provider in EXPECTED_PROVIDERS:
    results = (
        provider_filter_results[
            provider
        ]
    )

    check(
        len(results) > 0
        and
        all(
            result[
                "provider"
            ]
            ==
            provider

            for result
            in results
        ),
        (
            f"{provider} provider filter "
            "returns only "
            f"{provider}"
        ),
    )


# TEST 18
check(
    len(
        document_filter_results
    )
    > 0,
    "Specific document filter returns results",
)


# TEST 19
check(
    all(
        result[
            "document_id"
        ]
        ==
        AWS_SECURITY_DOCUMENT_ID

        for result
        in document_filter_results
    ),
    "Document filter stays inside one PDF",
)


# TEST 20
check(
    all(
        neighbor.metadata[
            "document_id"
        ]
        ==
        neighbor_test_result[
            "document_id"
        ]

        for neighbor
        in retrieved_neighbors
    ),
    "Neighbor expansion stays inside retrieved PDF",
)


# TEST 21
required_result_fields = {
    "provider",
    "document_id",
    "document_title",
    "source",
    "pdf_page",
    "document_chunk_number",
    "global_chunk_number",
}

check(
    all(
        required_result_fields.issubset(
            result.keys()
        )
        for result
        in general_results
    ),
    "Pinecone restores V12 document metadata",
)


# ============================================================
# FINAL RESULT
# ============================================================

tests_passed = sum(
    test_results
)

total_tests = len(
    test_results
)

accuracy = (
    tests_passed
    /
    total_tests
    *
    100
)


print()
print("=" * 70)
print("RAG V12 PINECONE MULTI-DOCUMENT TEST")
print("=" * 70)

print(
    "Index:",
    PINECONE_INDEX_NAME,
)

print(
    "Namespace:",
    PINECONE_NAMESPACE,
)

print(
    "Documents:",
    len(pdf_records),
)

print(
    "Pages:",
    len(pages),
)

print(
    "Chunks:",
    len(chunks),
)

print(
    "Embedding shape:",
    embeddings.shape,
)

print(
    f"Tests passed: "
    f"{tests_passed}/"
    f"{total_tests}"
)

print(
    f"Accuracy: "
    f"{accuracy:.1f}%"
)


if tests_passed == total_tests:
    print(
        "V12 STAGE 2 PINECONE MULTI-DOCUMENT INDEX: PASS ✅"
    )

else:
    print(
        "V12 STAGE 2 PINECONE MULTI-DOCUMENT INDEX: FAIL ❌"
    )
